# Amazon Stock Agent — Demo Execution

This notebook demonstrates the full end-to-end flow of the Stock Query Agent:
1. Authenticate via Cognito
2. Run 5 different queries (real-time, historical, RAG, combined)
3. Capture Langfuse trace IDs for observability

## Prerequisites

**Option A — Local (docker-compose):**
```bash
cp .env.example .env   # fill in your values
docker compose up --build
```

**Option B — AWS (Terraform):**
```bash
cd terraform/
cp terraform.tfvars.example terraform.tfvars  # edit values
terraform init && terraform apply
```

Then create a test user in Cognito:
```bash
aws cognito-idp admin-create-user \
  --user-pool-id <POOL_ID> \
  --username testuser@example.com \
  --user-attributes Name=email,Value=testuser@example.com Name=name,Value=TestUser \
  --temporary-password 'TempPass123!'
```

---
## 1. Setup

In [ ]:
import json
import os
import time
from datetime import datetime, timezone

import httpx
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

# ── Configuration ─────────────────────────────────────────────────
BASE_URL = os.getenv("API_URL", "http://localhost:8000")
USERNAME = os.getenv("TEST_USERNAME", "testuser@example.com")
PASSWORD = os.getenv("TEST_PASSWORD", "TempPass123!")

# Will be populated after auth
access_token = None
headers = {}

# Track execution times and trace IDs
results_log = []

print(f"API endpoint: {BASE_URL}")
print(f"Test user:    {USERNAME}")

In [ ]:
# Helper: send a streaming query and collect results
async def stream_query(question: str, label: str) -> dict:
    """Send a streaming POST /query request and print events in real time."""
    start = time.time()
    events = []
    trace_id = None
    final_answer = ""

    print(f"\n{'='*60}")
    print(f"Query: {question}")
    print(f"{'='*60}")

    async with httpx.AsyncClient(timeout=120.0) as client:
        async with client.stream(
            "POST",
            f"{BASE_URL}/query",
            headers=headers,
            json={"query": question, "stream": True},
        ) as resp:
            async for line in resp.aiter_lines():
                if not line.startswith("data: "):
                    continue
                event = json.loads(line[6:])
                events.append(event)
                trace_id = event.get("trace_id", trace_id)

                etype = event["type"]
                content = event.get("content", "")

                if etype == "thought":
                    print(f"  [Thought]  {content[:120]}")
                elif etype == "action":
                    print(f"  [Action]   {content}")
                elif etype == "observation":
                    print(f"  [Observe]  {content[:150]}...")
                elif etype == "final_answer":
                    final_answer = content
                    print(f"  [Answer]   {content[:200]}")
                elif etype == "done":
                    print("  [Done]")

    elapsed = round(time.time() - start, 2)
    result = {
        "label": label,
        "question": question,
        "trace_id": trace_id,
        "elapsed_s": elapsed,
        "events": events,
        "answer": final_answer,
    }
    results_log.append(result)
    print(f"\n  Trace ID: {trace_id}")
    print(f"  Time:     {elapsed}s")
    return result


# Helper: non-streaming query
async def simple_query(question: str, label: str) -> dict:
    """Send a non-streaming POST /query request."""
    start = time.time()
    async with httpx.AsyncClient(timeout=120.0) as client:
        r = await client.post(
            f"{BASE_URL}/query",
            headers=headers,
            json={"query": question, "stream": False},
        )
        data = r.json()

    elapsed = round(time.time() - start, 2)
    result = {
        "label": label,
        "question": question,
        "trace_id": data.get("trace_id"),
        "elapsed_s": elapsed,
        "answer": data.get("answer", ""),
        "sources": data.get("sources", []),
    }
    results_log.append(result)
    return result

---
## 2. Health Check & Authentication

In [ ]:
# Health check
async with httpx.AsyncClient() as client:
    r = await client.get(f"{BASE_URL}/health")
    health = r.json()
    print("Health:", health)
    assert health["status"] == "healthy", "API is not healthy!"

In [ ]:
# Authenticate with Cognito
async with httpx.AsyncClient() as client:
    r = await client.post(
        f"{BASE_URL}/auth/token",
        json={"username": USERNAME, "password": PASSWORD},
    )
    assert r.status_code == 200, f"Auth failed: {r.text}"
    tokens = r.json()
    access_token = tokens["access_token"]
    headers = {"Authorization": f"Bearer {access_token}"}

print("Authenticated successfully.")
print(f"Token type:  {tokens['token_type']}")
print(f"Expires in:  {tokens['expires_in']}s")

In [ ]:
# Verify user profile
async with httpx.AsyncClient() as client:
    r = await client.get(f"{BASE_URL}/auth/user", headers=headers)
    user_info = r.json()
    print("User info:")
    for k, v in user_info.items():
        print(f"  {k}: {v}")

---
## 3. Query Executions

### 3a. Real-time stock price

In [ ]:
result_a = await stream_query(
    "What is the stock price for Amazon right now?",
    label="3a_realtime_price",
)

In [ ]:
display(Markdown(f"### Agent Answer\n\n{result_a['answer']}"))

### 3b. Historical stock prices (Q4 2024)

In [ ]:
result_b = await stream_query(
    "What were the stock prices for Amazon in Q4 last year?",
    label="3b_historical_q4",
)

In [ ]:
# Plot historical prices if observation data is available
price_data = None
for ev in result_b.get("events", []):
    if ev["type"] == "observation":
        try:
            parsed = json.loads(ev["content"])
            if "prices" in parsed:
                price_data = parsed["prices"]
        except (json.JSONDecodeError, TypeError):
            pass

if price_data:
    df = pd.DataFrame(price_data)
    df["date"] = pd.to_datetime(df["date"])
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(df["date"], df["close"], label="Close", linewidth=2)
    ax.fill_between(df["date"], df["low"], df["high"], alpha=0.15, label="High-Low range")
    ax.set_title("AMZN — Q4 2024 Daily Close")
    ax.set_xlabel("Date")
    ax.set_ylabel("Price (USD)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No structured price data found in observations.")

display(Markdown(f"### Agent Answer\n\n{result_b['answer']}"))

### 3c. Combined: stock performance vs analyst reports

In [ ]:
result_c = await stream_query(
    "Compare Amazon's recent stock performance to what analysts predicted in their reports",
    label="3c_compare_reports",
)

In [ ]:
display(Markdown(f"### Agent Answer\n\n{result_c['answer']}"))

### 3d. RAG: current price + AI business insights

In [ ]:
result_d = await stream_query(
    "I'm researching AMZN give me the current price and any relevant information about their AI business",
    label="3d_price_plus_rag",
)

In [ ]:
display(Markdown(f"### Agent Answer\n\n{result_d['answer']}"))

### 3e. Pure knowledge base: office space in North America

In [ ]:
result_e = await stream_query(
    "What is the total amount of office space Amazon owned in North America in 2024?",
    label="3e_kb_office_space",
)

In [ ]:
display(Markdown(f"### Agent Answer\n\n{result_e['answer']}"))

---
## 4. Langfuse Traces

Each query generated a Langfuse trace. The trace IDs below link
directly to the Langfuse dashboard where you can inspect:
- Full agent reasoning chain
- Tool calls and their latencies
- Token usage per step

**Dashboard:** Check your Langfuse project at the host configured in `.env`.

In [ ]:
langfuse_host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")

print("Langfuse Trace Summary")
print("=" * 70)
for r in results_log:
    tid = r.get("trace_id", "N/A")
    print(f"\n  [{r['label']}]")
    print(f"  Question:  {r['question'][:80]}")
    print(f"  Trace ID:  {tid}")
    print(f"  Time:      {r['elapsed_s']}s")
    if tid and tid != "N/A":
        print(f"  Link:      {langfuse_host}/trace/{tid}")

    # Show which tools were invoked
    tools_used = [
        json.loads(e["content"]).get("tool", "")
        for e in r.get("events", [])
        if e.get("type") == "action"
    ]
    if tools_used:
        print(f"  Tools:     {', '.join(tools_used)}")

---
## 5. Execution Summary

In [ ]:
summary = pd.DataFrame([
    {
        "Query": r["label"],
        "Time (s)": r["elapsed_s"],
        "Trace ID": r.get("trace_id", "N/A"),
        "Answer (preview)": r.get("answer", "")[:80],
    }
    for r in results_log
])

display(summary)

total = sum(r["elapsed_s"] for r in results_log)
avg = total / len(results_log) if results_log else 0

print(f"\nTotal execution time: {total:.2f}s")
print(f"Average per query:    {avg:.2f}s")
print(f"Queries executed:     {len(results_log)}")
print(f"Timestamp:            {datetime.now(timezone.utc).isoformat()}")

---
## Conclusions

| Aspect | Result |
|--------|--------|
| **Cognito Auth** | Token-based auth flow working via `/auth/token` |
| **Real-time Prices** | `retrieve_realtime_stock_price` fetches live AMZN data via yfinance |
| **Historical Data** | `retrieve_historical_stock_price` returns OHLCV + trend analysis |
| **Knowledge Base (RAG)** | `search_financial_documents` retrieves relevant chunks from Amazon PDFs |
| **Streaming** | SSE streaming delivers incremental agent events to the client |
| **Observability** | Every query produces a Langfuse trace with full tool-call visibility |
| **Deployment** | Dockerized FastAPI runs on ECS Fargate via Terraform |